# Human Pose Estimation & Classification

This notebook uses **OpenCV** and **MediaPipe** to detect human body landmarks in images and a live webcam feed, then classifies the detected pose into one of three yoga/exercise poses (**Warrior II**, **T-Pose**, **Tree Pose**) using joint-angle heuristics.

**Before running:**
- Place your sample images inside the `images/` folder next to this notebook (or change `IMAGE_DIR` below).
- Run all cells in order — later cells depend on functions and variables defined earlier.

## 1. Imports & Configuration

In [ ]:
import os
import math
from time import time

import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt

# Folder containing the sample images used throughout this notebook.
# Update this path if your images live elsewhere.
IMAGE_DIR = "images"


## 2. Set Up the MediaPipe Pose Model

In [ ]:
# Initialize the MediaPipe Pose solution
mp_pose = mp.solutions.pose

# Set up the Pose model for static images (used for single-image inference below)
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.3, model_complexity=2)

# Utility for drawing landmarks on images
mp_drawing = mp.solutions.drawing_utils


## 3. Inspect Landmarks on a Sample Image

In [ ]:
# Read a sample image
sample_img = cv2.imread(os.path.join(IMAGE_DIR, "sample.jpg"))

plt.figure(figsize=[10, 10])
plt.title("Sample Image")
plt.axis('off')
plt.imshow(sample_img[:, :, ::-1])  # BGR -> RGB for correct display
plt.show()


In [ ]:
# MediaPipe expects RGB images, so convert before processing
results = pose.process(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))

# Print the first couple of landmarks (normalized coordinates) as a sanity check
if results.pose_landmarks:
    for i in range(2):
        landmark_name = mp_pose.PoseLandmark(i).name
        print(f'{landmark_name}:\n{results.pose_landmarks.landmark[i]}')


In [ ]:
# Convert normalized coordinates to pixel coordinates for the same two landmarks
image_height, image_width, _ = sample_img.shape

if results.pose_landmarks:
    for i in range(2):
        landmark = results.pose_landmarks.landmark[i]
        print(f'{mp_pose.PoseLandmark(i).name}:')
        print(f'x: {landmark.x * image_width}')
        print(f'y: {landmark.y * image_height}')
        print(f'z: {landmark.z * image_width}')
        print(f'visibility: {landmark.visibility}\n')


In [ ]:
# Draw the detected landmarks and skeleton connections on a copy of the image
img_copy = sample_img.copy()

if results.pose_landmarks:
    mp_drawing.draw_landmarks(
        image=img_copy,
        landmark_list=results.pose_landmarks,
        connections=mp_pose.POSE_CONNECTIONS
    )
    plt.figure(figsize=[10, 10])
    plt.title("Output")
    plt.axis('off')
    plt.imshow(img_copy[:, :, ::-1])
    plt.show()


In [ ]:
# Visualize the same pose as a 3D plot of world landmarks
mp_drawing.plot_landmarks(results.pose_world_landmarks, mp_pose.POSE_CONNECTIONS)


## 4. Reusable Pose Detection Function

In [ ]:
def detectPose(image, pose, display=True):
    """Run MediaPipe pose detection on a single image.

    Args:
        image: BGR image (as read by cv2.imread / cv2.VideoCapture).
        pose: A configured mediapipe.solutions.pose.Pose instance.
        display: If True, plot the original/output images and the 3D landmarks
                  instead of returning them.

    Returns:
        (output_image, landmarks) when display=False, otherwise None.
        landmarks is a list of (x, y, z) pixel-space coordinates.
    """
    output_image = image.copy()
    imageRGB = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(imageRGB)
    height, width, _ = image.shape
    landmarks = []

    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            image=output_image,
            landmark_list=results.pose_landmarks,
            connections=mp_pose.POSE_CONNECTIONS
        )
        for landmark in results.pose_landmarks.landmark:
            landmarks.append((
                int(landmark.x * width),
                int(landmark.y * height),
                landmark.z * width
            ))

    if display:
        plt.figure(figsize=[22, 22])
        plt.subplot(121); plt.imshow(image[:, :, ::-1]); plt.title("Original Image"); plt.axis('off')
        plt.subplot(122); plt.imshow(output_image[:, :, ::-1]); plt.title("Output Image"); plt.axis('off')
        mp_drawing.plot_landmarks(results.pose_world_landmarks, mp_pose.POSE_CONNECTIONS)
    else:
        return output_image, landmarks


### Try it on a few more sample images

In [ ]:
image = cv2.imread(os.path.join(IMAGE_DIR, "sample1.jpg"))
detectPose(image, pose, display=True)


In [ ]:
image = cv2.imread(os.path.join(IMAGE_DIR, "sample2.jpg"))
detectPose(image, pose, display=True)


In [ ]:
image = cv2.imread(os.path.join(IMAGE_DIR, "sample3.jpg"))
detectPose(image, pose, display=True)


## 5. Angle Calculation Utility

Pose classification works by measuring the angles between specific joints (e.g. shoulder-elbow-wrist) and comparing them against expected ranges for each pose.

In [ ]:
def calculateAngle(landmark1, landmark2, landmark3):
    """Calculate the angle (in degrees, 0-360) at landmark2, formed by the
    line segments landmark1-landmark2 and landmark3-landmark2."""
    x1, y1, _ = landmark1
    x2, y2, _ = landmark2
    x3, y3, _ = landmark3

    angle = math.degrees(
        math.atan2(y3 - y2, x3 - x2) - math.atan2(y1 - y2, x1 - x2)
    )
    if angle < 0:
        angle += 360
    return angle


In [ ]:
# Quick sanity check with arbitrary coordinates
angle = calculateAngle((558, 326, 0), (642, 333, 0), (718, 321, 0))
print(f'The calculated angle is {angle}')


## 6. Pose Classification

We classify three poses — **Warrior II**, **T-Pose**, and **Tree Pose** — using simple angle heuristics. This rule-based approach keeps things lightweight, but it doesn't scale well to many poses (a learned classifier, e.g. an MLP, would be a better fit for that — see the README's "Future Improvements").

In [ ]:
def classifyPose(landmarks, output_image, display=False):
    """Classify a pose as Warrior II, T-Pose, Tree Pose, or Unknown,
    based on joint angles, and annotate the output image with the label."""
    label = 'Unknown Pose'
    color = (0, 0, 255)  # Red for unrecognized poses; green once classified

    # Angles at the elbows
    left_elbow_angle = calculateAngle(
        landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value],
        landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value],
        landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]
    )
    right_elbow_angle = calculateAngle(
        landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value],
        landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value],
        landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value]
    )

    # Angles at the shoulders
    left_shoulder_angle = calculateAngle(
        landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value],
        landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value],
        landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
    )
    right_shoulder_angle = calculateAngle(
        landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value],
        landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value],
        landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value]
    )

    # Angles at the knees
    left_knee_angle = calculateAngle(
        landmarks[mp_pose.PoseLandmark.LEFT_HIP.value],
        landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value],
        landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value]
    )
    right_knee_angle = calculateAngle(
        landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value],
        landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value],
        landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value]
    )

    # --- Warrior II Pose / T-Pose: both arms roughly straight and raised to shoulder height ---
    if 165 < left_elbow_angle < 195 and 165 < right_elbow_angle < 195:
        if 80 < left_shoulder_angle < 110 and 80 < right_shoulder_angle < 110:

            # Warrior II: one leg straight, the other bent at the knee
            if (165 < left_knee_angle < 195) or (165 < right_knee_angle < 195):
                if (90 < left_knee_angle < 120) or (90 < right_knee_angle < 120):
                    label = 'Warrior II Pose'

            # T-Pose: both legs straight
            if 160 < left_knee_angle < 195 and 160 < right_knee_angle < 195:
                label = 'T Pose'

    # --- Tree Pose: one leg straight, the other knee bent and lifted ---
    if (165 < left_knee_angle < 195) or (165 < right_knee_angle < 195):
        if (315 < left_knee_angle < 335) or (25 < right_knee_angle < 45):
            label = 'Tree Pose'

    if label != 'Unknown Pose':
        color = (0, 255, 0)

    cv2.putText(output_image, label, (10, 30), cv2.FONT_HERSHEY_PLAIN, 2, color, 2)

    if display:
        plt.figure(figsize=[10, 10])
        plt.imshow(output_image[:, :, ::-1])
        plt.title("Output Image")
        plt.axis('off')
    else:
        return output_image, label


### Test classification on sample images

In [ ]:
# Classify Warrior II pose
image = cv2.imread(os.path.join(IMAGE_DIR, "warriorIIpose.jpg"))
output_image, landmarks = detectPose(image, pose, display=False)
if landmarks:
    classifyPose(landmarks, output_image, display=True)


In [ ]:
# Classify Tree pose
image = cv2.imread(os.path.join(IMAGE_DIR, "treepose3.jpg"))
output_image, landmarks = detectPose(
    image,
    mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5, model_complexity=0),
    display=False
)
if landmarks:
    classifyPose(landmarks, output_image, display=True)


In [ ]:
# Classify T pose
image = cv2.imread(os.path.join(IMAGE_DIR, "tpose.jpg"))
output_image, landmarks = detectPose(image, pose, display=False)
if landmarks:
    classifyPose(landmarks, output_image, display=True)


In [ ]:
# Classify an unrecognized / unknown pose
image = cv2.imread(os.path.join(IMAGE_DIR, "unknown.jpg"))
output_image, landmarks = detectPose(image, pose, display=False)
if landmarks:
    classifyPose(landmarks, output_image, display=True)


## 7. Real-Time Pose Classification via Webcam

This opens your default webcam, detects and classifies the pose in each frame, and overlays the result live. Press **Esc** to quit.

> **Note:** This cell requires a connected webcam and a local environment with a display (it will not work in a headless or purely browser-based notebook environment).

In [ ]:
pose_video = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, model_complexity=1)

camera_video = cv2.VideoCapture(0)
camera_video.set(3, 1280)
camera_video.set(4, 960)

cv2.namedWindow('Pose Classification', cv2.WINDOW_NORMAL)

while camera_video.isOpened():
    ok, frame = camera_video.read()
    if not ok:
        continue

    frame = cv2.flip(frame, 1)
    frame_height, frame_width, _ = frame.shape
    frame = cv2.resize(frame, (int(frame_width * (640 / frame_height)), 640))

    frame, landmarks = detectPose(frame, pose_video, display=False)
    if landmarks:
        frame, _ = classifyPose(landmarks, frame, display=False)

    cv2.imshow('Pose Classification', frame)

    k = cv2.waitKey(1) & 0xFF
    if k == 27:  # Esc key
        break

camera_video.release()
cv2.destroyAllWindows()
